# Split review

Do the folds and the test set get a similar mix of cases? The splits are stratified
on the patients' strata (see `docs/data-pipeline.md`): with or without a mass, a
small mass (under 100 mm²) or not, dense breasts (ACR 3 or 4) or not. This
notebook shows, for every fold and the test set, the strata, the mass sizes and the
breast densities.

It compares two splits files made from the same manifest: the current
`splits.json` and, if it is there, the earlier one made before the stratification
by strata, saved as `splits_v1.json` (copy the old `splits.json` to that name before
making the new one). Both are described with the same thresholds, those of the new
file. Only counts and sizes are shown, no images. Outputs are stripped before
commit (`nbstripout`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mammo_seg.data.manifest import read_manifest
from mammo_seg.data.schema import LesionType
from mammo_seg.data.splits import (
    check_manifest,
    manifest_fingerprint,
    read_splits,
    select_images,
)
from mammo_seg.data.strata import area_mm2, group_name, patient_profiles


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
PROCESSED = ROOT / "data" / "processed"
NEW_SPLITS = PROCESSED / "splits.json"
OLD_SPLITS = PROCESSED / "splits_v1.json"  # None to show the new splits only
manifest = read_manifest(PROCESSED / "manifest.json")

## The splits

The new file is read and validated as usual. The old one has another format
version, so only its patient lists are read, after checking that it was made from
the same manifest.

In [ ]:
def sets_of(path: Path) -> dict[str, str]:
    """The set of every patient (``fold_<i>`` or ``test``) in a splits file of any
    version; ValueError if it was made from another manifest."""
    data = json.loads(path.read_text())
    if data["manifest_fingerprint"] != manifest_fingerprint(manifest):
        raise ValueError(f"{path} was made from another manifest")
    splits = data["splits"]
    sets = {patient: "test" for patient in splits["test_patients"]}
    for index, fold in enumerate(splits["folds"]):
        sets.update({patient: f"fold_{index}" for patient in fold})
    return sets


new_file = read_splits(NEW_SPLITS)
check_manifest(new_file, manifest)
SPLITS = {"new": sets_of(NEW_SPLITS)}
if OLD_SPLITS is not None and OLD_SPLITS.exists():
    SPLITS = {"old": sets_of(OLD_SPLITS), **SPLITS}
else:
    print(f"no old splits at {OLD_SPLITS}: showing the new splits only")
SETS = sorted(set(SPLITS["new"].values()), key=lambda name: (name == "test", name))
THRESHOLDS = new_file.stratification
print("stratification:", THRESHOLDS)
for name, sets in SPLITS.items():
    print(f"{name}: {len(sets)} patients in {len(set(sets.values()))} sets")

In [ ]:
images = select_images(manifest)
profiles = patient_profiles(images, THRESHOLDS)
patients = pd.DataFrame(
    {
        "patient_id": [p.patient_id for p in profiles],
        "stratum": [group_name(p.group) for p in profiles],
        "has_mass": [p.has_mass for p in profiles],
        "dense": [p.dense for p in profiles],
        "small_mass": [p.small_mass for p in profiles],
    }
)
image_table = pd.DataFrame(
    {
        "patient_id": [image.patient_id for image in images],
        "acr": [image.acr for image in images],
        "normal": [image.birads == "1" for image in images],
        "has_mass": [image.has_mass for image in images],
    }
)
masses = pd.DataFrame(
    [
        {
            "patient_id": image.patient_id,
            "area_mm2": area_mm2(roi, THRESHOLDS.pixel_size_mm),
        }
        for image in images
        for roi in image.rois
        if roi.lesion_type is LesionType.MASS
    ]
)


def with_set(frame: pd.DataFrame, split: str) -> pd.DataFrame:
    return frame.assign(set=frame.patient_id.map(SPLITS[split]))

## Counts per set

Per set: patients, those with a mass, a dense breast or a small mass, images, images
with a mass and normal images (BI-RADS 1), and the masses with their median area.
In a good split each column is about proportional to the size of the set.

In [ ]:
def summary(split: str) -> pd.DataFrame:
    p = with_set(patients, split).groupby("set")
    i = with_set(image_table, split).groupby("set")
    m = with_set(masses, split).groupby("set")
    table = pd.DataFrame(
        {
            "patients": p.size(),
            "with mass": p.has_mass.sum(),
            "dense": p.dense.sum(),
            "small mass": p.small_mass.sum(),
            "images": i.size(),
            "images with mass": i.has_mass.sum(),
            "normal images": i.normal.sum(),
            "masses": m.size(),
            "median mass mm²": m.area_mm2.median().round(0),
            f"masses < {THRESHOLDS.small_mass_mm2:g} mm²": m.area_mm2.apply(
                lambda areas: int((areas < THRESHOLDS.small_mass_mm2).sum())
            ),
        }
    )
    return table.loc[SETS]


for split in SPLITS:
    print(f"\n{split} splits")
    display(summary(split))

## Strata

The share of the patients of every set in each stratum. In a good split the bars of
all sets look alike.

In [ ]:
STRATA = sorted(patients.stratum.unique())
COLOURS = dict(zip(STRATA, plt.get_cmap("tab10").colors, strict=False))

fig, axes = plt.subplots(
    1, len(SPLITS), figsize=(7 * len(SPLITS), 4), sharey=True, squeeze=False
)
for ax, split in zip(axes[0], SPLITS, strict=True):
    counts = pd.crosstab(with_set(patients, split).set, patients.stratum).loc[SETS]
    shares = counts.div(counts.sum(axis=1), axis=0)
    left = np.zeros(len(SETS))
    for stratum in STRATA:
        values = shares.get(stratum, pd.Series(0, index=SETS)).to_numpy()
        ax.barh(SETS, values, left=left, color=COLOURS[stratum], label=stratum)
        left += values
    ax.set_title(f"{split} splits: patients per stratum")
    ax.set_xlabel("share of the patients of the set")
axes[0, 0].invert_yaxis()  # the axes are shared: invert once
axes[0, -1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Mass sizes

Every mass of every set, in mm² on a log scale, with the median (black bar) and the
small-mass threshold (dashed). The same mass in the CC and the MLO view counts twice.
In a good split the sets cover a similar range.

In [ ]:
rng = np.random.default_rng(0)  # only spreads the points sideways
fig, axes = plt.subplots(
    1, len(SPLITS), figsize=(7 * len(SPLITS), 4), sharey=True, squeeze=False
)
for ax, split in zip(axes[0], SPLITS, strict=True):
    table = with_set(masses, split)
    for position, name in enumerate(SETS):
        areas = table.area_mm2[table.set == name].to_numpy()
        ax.scatter(
            position + rng.uniform(-0.15, 0.15, len(areas)), areas, s=12, alpha=0.7
        )
        if len(areas):
            ax.hlines(np.median(areas), position - 0.3, position + 0.3, color="black")
    ax.axhline(THRESHOLDS.small_mass_mm2, color="grey", linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_xticks(range(len(SETS)), SETS)
    ax.set_ylabel("mass area (mm²)")
    ax.set_title(f"{split} splits: mass sizes")
plt.tight_layout()
plt.show()

## Breast density

The share of the images of every set at each ACR density (one image has none).

In [ ]:
DENSITIES = ["1", "2", "3", "4", "none"]
DENSITY_COLOURS = dict(
    zip(DENSITIES, plt.get_cmap("Blues")(np.linspace(0.25, 1, 4)), strict=False)
)
DENSITY_COLOURS["none"] = "lightgrey"

fig, axes = plt.subplots(len(SPLITS), 1, figsize=(8, 3 * len(SPLITS)), squeeze=False)
for (ax,), split in zip(axes, SPLITS, strict=True):
    table = with_set(image_table, split)
    density = table.acr.map(lambda acr: "none" if pd.isna(acr) else str(int(acr)))
    counts = pd.crosstab(table.set, density).reindex(
        index=SETS, columns=DENSITIES, fill_value=0
    )
    shares = counts.div(counts.sum(axis=1), axis=0)
    left = np.zeros(len(SETS))
    for value in DENSITIES:
        ax.barh(
            SETS,
            shares[value],
            left=left,
            color=DENSITY_COLOURS[value],
            label=f"ACR {value}",
        )
        left += shares[value].to_numpy()
    ax.set_title(f"{split} splits: images per ACR density")
    ax.set_xlabel("share of the images of the set")
    ax.invert_yaxis()
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()